# WikiText tutorial with Hugging Face tokenizers

The [WikiText data tutorial](https://docs.fast.ai/tutorial.wikitext.html) builds a language model with the mid-level API: `Tokenizer`, `Numericalize`, `Datasets` and `LMDataLoader`. This dialog follows the same steps with `fastai.text.tokenizer`. The tokenizer is GPT-2's byte-level BPE, so there is no vocabulary to build and no `Numericalize`. The texts are Simple English Wikipedia articles from the `answerdotai/simplewiki` dataset.

In [ ]:
from fastai.text.all import *
from fastai.text.tokenizer import *

/Users/hoa/aai-ws/.venv/lib/python3.13/site-packages/torch/jit/_script.py:1491: FutureWarning: `torch.jit.script` is deprecated. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


## Data

`tokenize_dataset` downloads the dataset, encodes every article with GPT-2 and saves the token cache on the first call. Later calls load the cache. `split_docs` gives one array of token ids per article.

In [ ]:
flat, offsets, tok = tokenize_dataset('answerdotai/simplewiki', itemgetter('md'), 'articles')
docs = split_docs(flat, offsets)
len(docs), len(flat)

(270509, 183842357)

In [ ]:
tok.decode(docs[0][:50])

'**April** (Apr.) is the fourth month of the year in the Julian and Gregorian calendars, and comes between March and May. It is one of four months to have 30 days.\n\nApril always begins on the same day of the'

## Datasets

`HfTokenizer` is the one transform of the pipeline. It adds `<bos>` and `<eos>` around each article and decodes a batch back to text. `Datasets` with `dl_type=LMDataLoader` is what `TextBlock(is_lm=True)` builds in the tutorial.

In [ ]:
tokenizer = HfTokenizer(tok)
splits = RandomSplitter(valid_pct=0.1, seed=42)(docs)
dsets = Datasets(docs, [tokenizer], splits=splits, dl_type=LMDataLoader)

In [ ]:
bs,sl = 8,72
dls = dsets.dataloaders(bs=bs, seq_len=sl)
dls.show_batch()

<bos>**Ioannis Tassias** (1 October 1958 – 15 November 2020) was a Greek Orthodox bishop. He was the Metropolitan of Langadas as **Ioannis of Langada** from 2010 until his death in 2020. He was born in Thessaloniki, Greece.

On 13 November 2020, Tassias was hospitalized with
ie is divided into Brigades consisting of 6 to 40 operators. A brigade performs observation and may give assistance to citizens in regulated periods, during both daytime and nighttime.

### *Mobile Gendarmerie*

The Mobile Gendarmerie has several tasks, which include riot control, security of significant places (like embassies), conducting OPEX (
-align:center; border:1px \#aaf solid;"
! rowspan\="2"\| \#\|Number
! rowspan\="2" class\="unsortable"\| Portrait
! rowspan\="2" style\="width:14em" \| Name\
[(
|**1. DFC Leoben**
\|0-3
\|-
\|Bad Leonfelden
\|**DFC Ostbahn XI** 
\|0-9
\|-
\|Blaue Elf Linz
\|**USC Landhaus**  
\|
div data-arg>1965</div><div data-arg>04</div><div data-arg>14</div><div data-arg>1966</div><div

## Model

The AWD-LSTM configuration of the tutorial, with its dropout probabilities. `len(dls.vocab)` is the tokenizer's vocabulary size, read through the pipeline.

In [ ]:
config = awd_lstm_lm_config.copy()
config.update({'input_p': 0.6, 'output_p': 0.4, 'weight_p': 0.5, 'embed_p': 0.1, 'hidden_p': 0.2})
model = get_language_model(AWD_LSTM, len(dls.vocab), config=config)

In [ ]:
opt_func = partial(Adam, wd=0.1, eps=1e-7)
cbs = [MixedPrecision(), GradientClip(0.1)] + rnn_cbs(alpha=2, beta=1)
learn = Learner(dls, model, loss_func=CrossEntropyLossFlat(), opt_func=opt_func, cbs=cbs, metrics=[accuracy, Perplexity()])

## Training

In [ ]:
learn.fit_one_cycle(1, 5e-3, moms=(0.8,0.7,0.8), div=10)